# ReciteRight model server

This notebook loads the Quran speech model and runs a small web service. The prototype page sends short recordings to it and gets back the words, with vowel marks.

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.

**Run the cells in order.** When the last cell finishes, it prints a **server address**. Copy that address into the prototype page.

The address changes each time you run the notebook. Anyone with the address can use the server while it's running, so only share it with people testing the prototype.

In [ ]:
!pip install -q transformers peft librosa fastapi uvicorn python-multipart

In [ ]:
import torch
import librosa
from peft import PeftConfig, PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor, GenerationConfig

ADAPTER_ID = 'aboalaa1472/whisper-quran-lora-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    config = PeftConfig.from_pretrained(ADAPTER_ID)
    base_id = config.base_model_name_or_path
    processor = WhisperProcessor.from_pretrained(base_id)
    base = WhisperForConditionalGeneration.from_pretrained(base_id)
    model = PeftModel.from_pretrained(base, ADAPTER_ID)
except Exception as e:
    print('Not an adapter, loading as full model. Reason:', type(e).__name__)
    processor = WhisperProcessor.from_pretrained(ADAPTER_ID)
    model = WhisperForConditionalGeneration.from_pretrained(ADAPTER_ID)

model = model.to(device).eval()
model.generation_config = GenerationConfig.from_pretrained('openai/whisper-large-v3-turbo')
print('Model loaded on', device)

def transcribe_file(path):
    audio, _ = librosa.load(path, sr=16000, mono=True)
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
    feats = inputs.input_features.to(device, dtype=model.dtype)
    with torch.no_grad():
        ids = model.generate(feats, language='ar', task='transcribe', max_new_tokens=200)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

In [ ]:
import os
import subprocess
import tempfile
import threading
import time
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/')
def health():
    return {'status': 'ok'}

@app.post('/transcribe')
def transcribe(audio: UploadFile = File(...)):
    data = audio.file.read()
    with tempfile.TemporaryDirectory() as d:
        src = os.path.join(d, 'in.webm')
        dst = os.path.join(d, 'out.wav')
        with open(src, 'wb') as f:
            f.write(data)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-ar', '16000', '-ac', '1', dst], check=True)
        text = transcribe_file(dst)
    return {'text': text}

threading.Thread(
    target=uvicorn.run,
    kwargs={'app': app, 'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'},
    daemon=True,
).start()
time.sleep(3)
print('Local service running on port 8000.')

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import re

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
server_address = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        server_address = m.group(0)
        break

# Keep reading the tunnel's output so it never stalls
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

print('Server address to paste into the prototype page:')
print(server_address)